In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Metin temizleme için stop-words (etkisiz kelimeler) indirelim
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# Veriyi yükle (Dosya adının doğru olduğundan emin ol)
df = pd.read_csv('/IMDB Dataset.csv')

# İlk birkaç satıra göz atalım
print("Veri Kümesi İlk 5 Satır:")
print(df.head())

# Sınıfların dağılımını kontrol edelim (Dengeli mi?)
print("\nSınıf Dağılımı:")
print(df['sentiment'].value_counts())

Veri Kümesi İlk 5 Satır:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Sınıf Dağılımı:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [6]:
def clean_text(text):
    # HTML etiketlerini kaldır (<br /> gibi)
    text = re.sub(r'<br\s*/?>', ' ', text)
    # Alfanumerik olmayan karakterleri ve noktalama işaretlerini kaldır
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Küçük harfe çevir
    text = text.lower()
    # Stop words (the, is, at vb.) temizle ve boşlukları düzenle
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

# Temizleme fonksiyonunu tüm verilere uygulayalım (Biraz zaman alabilir)
print("Metinler temizleniyor, lütfen bekleyin...")
df['cleaned_review'] = df['review'].apply(clean_text)
print("Temizleme tamamlandı!")

Metinler temizleniyor, lütfen bekleyin...
Temizleme tamamlandı!


In [7]:
# Duyguları sayısallaştır: positive -> 1, negative -> 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# X (Girdi) ve y (Çıktı) değişkenlerini belirle
X = df['cleaned_review']
y = df['label']

# %80 Eğitim, %20 Test olacak şekilde ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Eğitim veri seti boyutu: {X_train.shape[0]}")
print(f"Test veri seti boyutu: {X_test.shape[0]}")

Eğitim veri seti boyutu: 40000
Test veri seti boyutu: 10000


In [8]:
# TF-IDF Vektörleştiriciyi tanımla
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))

# Eğitim verisini fit et ve dönüştür
X_train_tfidf = tfidf.fit_transform(X_train)

# Test verisini sadece dönüştür (fit etme!)
X_test_tfidf = tfidf.transform(X_test)

print("Metinler başarıyla sayısal vektörlere dönüştürüldü.")

Metinler başarıyla sayısal vektörlere dönüştürüldü.


In [9]:
# Modeli tanımla (C=10 regülasyonu güçlendirerek doğruluğu artırır)
model = LogisticRegression(C=10, max_iter=1000, n_jobs=-1)

# Modeli eğit
model.fit(X_train_tfidf, y_train)
print("Model eğitimi tamamlandı!")

Model eğitimi tamamlandı!


In [10]:
# Test verisi üzerinde tahmin yap
y_pred = model.predict(X_test_tfidf)

# Doğruluk skoru
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Doğruluğu (Accuracy): {accuracy:.4f}\n")

# Detaylı rapor
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Model Doğruluğu (Accuracy): 0.9066

Sınıflandırma Raporu:
              precision    recall  f1-score   support

    Negative       0.91      0.90      0.91      5000
    Positive       0.90      0.91      0.91      5000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



In [11]:
def predict_my_review(custom_reviews):
    # Yazılan yorumları temizle ve vektörleştir
    cleaned_custom = [clean_text(r) for r in custom_reviews]
    vectorized_custom = tfidf.transform(cleaned_custom)

    # Tahmin yap
    predictions = model.predict(vectorized_custom)
    probabilities = model.predict_proba(vectorized_custom)

    for review, pred, prob in zip(custom_reviews, predictions, probabilities):
        sentiment = "POZİTİF 😊" if pred == 1 else "NEGATİF 😞"
        confidence = prob[1] if pred == 1 else prob[0]
        print(r"---")
        print(f"Yorum: '{review}'")
        print(f"Tahmin: {sentiment} (%{confidence*100:.2f} güven oranı)")

# Test etmek istediğin cümleleri buraya yazabilirsin:
my_reviews = [
    "I absolutely loved this movie! The acting was incredible and the plot kept me on the edge of my seat.",
    "What a waste of time. The story made no sense and the characters were extremely annoying.",
    "It wasn't a masterpiece, but it was still quite entertaining and fun to watch with family."
]

predict_my_review(my_reviews)

---
Yorum: 'I absolutely loved this movie! The acting was incredible and the plot kept me on the edge of my seat.'
Tahmin: POZİTİF 😊 (%99.95 güven oranı)
---
Yorum: 'What a waste of time. The story made no sense and the characters were extremely annoying.'
Tahmin: NEGATİF 😞 (%99.95 güven oranı)
---
Yorum: 'It wasn't a masterpiece, but it was still quite entertaining and fun to watch with family.'
Tahmin: POZİTİF 😊 (%99.45 güven oranı)
